# Image Restoration Techniques (RGB)

This notebook demonstrates various image restoration techniques on RGB images:
- Arithmetic Mean Filter
- Median Filter
- Wiener Filter
- Pseudo-Inverse Filter
- Trimmed Average Filter
- Gaussian Filter

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import wiener

## Helper Functions

In [2]:
def arithmetic_mean_filter(image, kernel_size=3):
    """Apply arithmetic mean filter to each channel"""
    if len(image.shape) == 2:  # If grayscale
        return cv2.blur(image, (kernel_size, kernel_size))
    else:  # If RGB
        channels = cv2.split(image)
        restored_channels = [cv2.blur(channel, (kernel_size, kernel_size)) for channel in channels]
        return cv2.merge(restored_channels)

def median_filter(image, kernel_size=3):
    """Apply median filter to each channel"""
    if len(image.shape) == 2:  # If grayscale
        return cv2.medianBlur(image, kernel_size)
    else:  # If RGB
        channels = cv2.split(image)
        restored_channels = [cv2.medianBlur(channel, kernel_size) for channel in channels]
        return cv2.merge(restored_channels)

def wiener_filter(image, kernel_size=3):
    """Apply Wiener filter to each channel"""
    if len(image.shape) == 2:  # If grayscale
        image_float = image.astype(np.float32)
        restored = wiener(image_float, (kernel_size, kernel_size))
        return np.clip(restored, 0, 255).astype(np.uint8)
    else:  # If RGB
        channels = cv2.split(image)
        restored_channels = []
        for channel in channels:
            channel_float = channel.astype(np.float32)
            restored = wiener(channel_float, (kernel_size, kernel_size))
            restored_channels.append(np.clip(restored, 0, 255).astype(np.uint8))
        return cv2.merge(restored_channels)

def pseudo_inverse_filter(image, kernel_size=3):
    """Apply pseudo-inverse filter to each channel"""
    kernel = np.ones((kernel_size, kernel_size)) / (kernel_size * kernel_size)
    
    if len(image.shape) == 2:  # If grayscale
        image_fft = np.fft.fft2(image)
        kernel_fft = np.fft.fft2(kernel, s=image.shape)
        epsilon = 1e-6
        restored_fft = image_fft / (kernel_fft + epsilon)
        restored = np.fft.ifft2(restored_fft)
        return np.abs(restored).astype(np.uint8)
    else:  # If RGB
        channels = cv2.split(image)
        restored_channels = []
        for channel in channels:
            channel_fft = np.fft.fft2(channel)
            kernel_fft = np.fft.fft2(kernel, s=channel.shape)
            epsilon = 1e-6
            restored_fft = channel_fft / (kernel_fft + epsilon)
            restored = np.fft.ifft2(restored_fft)
            restored_channels.append(np.abs(restored).astype(np.uint8))
        return cv2.merge(restored_channels)

def trimmed_average_filter(image, kernel_size=3, trim_percent=20):
    """Apply trimmed average filter to each channel"""
    pad = kernel_size // 2
    
    if len(image.shape) == 2:  # If grayscale
        padded = cv2.copyMakeBorder(image, pad, pad, pad, pad, cv2.BORDER_REFLECT)
        result = np.zeros_like(image)
        
        for i in range(pad, padded.shape[0] - pad):
            for j in range(pad, padded.shape[1] - pad):
                window = padded[i-pad:i+pad+1, j-pad:j+pad+1]
                sorted_window = np.sort(window.flatten())
                trim = int(len(sorted_window) * trim_percent / 100)
                result[i-pad, j-pad] = np.mean(sorted_window[trim:-trim])
        
        return result.astype(np.uint8)
    else:  # If RGB
        channels = cv2.split(image)
        restored_channels = []
        for channel in channels:
            padded = cv2.copyMakeBorder(channel, pad, pad, pad, pad, cv2.BORDER_REFLECT)
            result = np.zeros_like(channel)
            
            for i in range(pad, padded.shape[0] - pad):
                for j in range(pad, padded.shape[1] - pad):
                    window = padded[i-pad:i+pad+1, j-pad:j+pad+1]
                    sorted_window = np.sort(window.flatten())
                    trim = int(len(sorted_window) * trim_percent / 100)
                    result[i-pad, j-pad] = np.mean(sorted_window[trim:-trim])
            
            restored_channels.append(result.astype(np.uint8))
        return cv2.merge(restored_channels)

def gaussian_filter(image, kernel_size=3, sigma=1.0):
    """Apply Gaussian filter to each channel"""
    if len(image.shape) == 2:  # If grayscale
        return cv2.GaussianBlur(image, (kernel_size, kernel_size), sigma)
    else:  # If RGB
        channels = cv2.split(image)
        restored_channels = [cv2.GaussianBlur(channel, (kernel_size, kernel_size), sigma) for channel in channels]
        return cv2.merge(restored_channels)

## Load and Process Images

In [ ]:
# Dictionary of noisy images
noisy_images = {
    'Gaussian': 'Images/Noisy_Images/gaussian_noise.jpg',
    'Salt & Pepper': 'Images/Noisy_Images/salt_pepper_noise.jpg',
    'Rayleigh': 'Images/Noisy_Images/rayleigh_noise.jpg',
    'Motion Blur': 'Images/Noisy_Images/motion_blur.jpg',
    'Exponential': 'Images/Noisy_Images/exponential_noise.jpg'
}

# Parameters for restoration
kernel_size = 3
sigma = 1.0
trim_percent = 20

# Process each noisy image
for noise_type, image_path in noisy_images.items():
    print(f"\nProcessing {noise_type} noise...")
    
    # Read the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not read {image_path}")
        continue
    
    # Convert BGR to RGB for display
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Apply all restoration techniques
    techniques = {
        'Arithmetic Mean': arithmetic_mean_filter(image_rgb, kernel_size),
        'Median': median_filter(image_rgb, kernel_size),
        'Wiener': wiener_filter(image_rgb, kernel_size),
        'Pseudo Inverse': pseudo_inverse_filter(image_rgb, kernel_size),
        'Trimmed Average': trimmed_average_filter(image_rgb, kernel_size, trim_percent),
        'Gaussian': gaussian_filter(image_rgb, kernel_size, sigma)
    }
    
    # Display results
    plt.figure(figsize=(15, 10))
    
    # Original image
    plt.subplot(241)
    plt.imshow(image_rgb)
    plt.title('Original')
    plt.axis('off')
    
    # Restored images
    for i, (technique, restored) in enumerate(techniques.items()):
        plt.subplot(242 + i)
        plt.imshow(restored)
        plt.title(technique)
        plt.axis('off')
    
    plt.suptitle(f'{noise_type} Noise Restoration', fontsize=16)
    plt.tight_layout()
    plt.show()